# DengAI-UNT (Fase 2 · Sesión 2): Nested Walk-Forward CV con Optuna y GPU

Ejecuta la **Validación Cruzada Anidada temporal (`5 outer folds × 3 inner folds`)** sobre los datos **100 % reales** (MINSA + NASA POWER, 2015–2024) excluyendo el bloque intocable de las últimas **26 semanas de prueba** (`2024-06-24` en adelante).

### Garantías de este notebook para Colab Pro
- **Cero pérdida ante desconexiones:** monta **Google Drive** y guarda `nested_cv_results.json` + telemetría directamente en Drive tras cada fold externo.
- **Reanudación automática (`--resume`):** si Colab se reinicia o se detiene, basta volver a ejecutar la celda de entrenamiento para continuar exactamente en el fold pendiente.
- **Preserva el entorno CUDA de Colab:** instala únicamente los paquetes faltantes sin sobreescribir `torch` ni `numpy`.
- **Métricas completas del Anexo A:** `MAE`, `RMSE`, `MASE`, `WIS`, `coverage`, `AUC_PR`, `AUC_ROC`, `F1`, `MCC`, `sensitivity`, `specificity`, `Brier` y `tiempo_s`, con Media, DE, Mín, Máx e IC 95 %.
- **Descarga automática:** al finalizar exporta CSVs de resumen, JSON completo y telemetría en un `.zip`.

In [ ]:
# 1. Verificar entorno Python, PyTorch y GPU asignada en Google Colab
import sys
import torch

print(f"Python: {sys.version.split()[0]}")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA disponible: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA runtime: {torch.version.cuda}")
    print(f"GPU activa: {torch.cuda.get_device_name(0)}")
    vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
    print(f"VRAM total: {vram_gb:.2f} GB")
else:
    print("AVISO: Estás en CPU. En Colab ve a: Entorno de ejecución -> Cambiar tipo de entorno de ejecución -> GPU (T4, L4 o A100).")

In [ ]:
# 2. Montar Google Drive para persistencia incremental (protege contra desconexiones)
import os
from pathlib import Path

IN_COLAB = os.path.isdir("/content")
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_DIR = Path("/content/drive/MyDrive/DengAI-UNT-Phase2")
else:
    DRIVE_DIR = Path.cwd() / "data" / "reports" / "phase2"

DRIVE_DIR.mkdir(parents=True, exist_ok=True)
(DRIVE_DIR / "telemetry").mkdir(parents=True, exist_ok=True)

OUTPUT_JSON = DRIVE_DIR / "nested_cv_results.json"
TELEMETRY_DIR = DRIVE_DIR / "telemetry"
print(f"Carpeta persistente en Drive: {DRIVE_DIR}")
print(f"Resultados incrementales: {OUTPUT_JSON}")

In [ ]:
# 3. Cargar código del proyecto (desde ZIP en Drive o desde GitHub) e instalar dependencias
import shutil
import subprocess
import zipfile
from pathlib import Path

REPO_URL = "https://github.com/JosephBanGH/DengAI-UNT.git"
REPO_DIR = Path("/content/DengAI-UNT") if os.path.isdir("/content") else Path.cwd()
BUNDLE_ZIP = DRIVE_DIR / "dengai_nested_cv_bundle.zip"

if BUNDLE_ZIP.exists():
    print(f"Descomprimiendo snapshot local desde Drive: {BUNDLE_ZIP}")
    REPO_DIR.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(BUNDLE_ZIP, "r") as zf:
        zf.extractall(REPO_DIR)
else:
    if not (REPO_DIR / "backend").is_dir():
        print(f"Clonando repositorio {REPO_URL}...")
        subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
    else:
        print("Actualizando repositorio desde origin/main...")
        subprocess.run(["git", "-C", str(REPO_DIR), "pull", "origin", "main"], check=True)

os.chdir(REPO_DIR)
assert (REPO_DIR / "training" / "run_nested_cv.py").exists(), (
    "No se encontró training/run_nested_cv.py. Sube 'dengai_nested_cv_bundle.zip' a tu carpeta " 
    f"{DRIVE_DIR} en Google Drive o haz git push antes de clonar."
)

# Instalar solo librerías faltantes preservando PyTorch + CUDA y NumPy preinstalados en Colab
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "optuna", "pydantic-settings", "xgboost", "httpx", "bcrypt", "PyJWT", "sqlalchemy"],
    check=True,
)

import importlib.metadata as md
for pkg in ("torch", "xgboost", "optuna", "scikit-learn", "scipy", "numpy", "pandas", "sqlalchemy"):
    try:
        print(f"  - {pkg}: {md.version(pkg)}")
    except md.PackageNotFoundError:
        print(f"  - {pkg}: NO ENCONTRADO")

In [ ]:
# 4. Asegurar base de datos 100 % REAL (MINSA + NASA POWER, sin caer en datos sintéticos)
import os
import sys
os.environ.setdefault("BASE_URL", "http://127.0.0.1:8000")
if str(REPO_DIR / "backend") not in sys.path:
    sys.path.insert(0, str(REPO_DIR / "backend"))

local_db = REPO_DIR / "data" / "dengai.db"
drive_db = DRIVE_DIR / "dengai.db"
(REPO_DIR / "data").mkdir(parents=True, exist_ok=True)

if not local_db.exists() and drive_db.exists():
    print(f"Copiando base de datos desde Drive: {drive_db} -> {local_db}")
    shutil.copy2(drive_db, local_db)

from sqlalchemy import func, select
from app.db.base import Base
from app.db.seed import ensure_data_sources, ensure_geo_units
from app.db.session import SessionLocal, engine
from app.ml.features import load_panel
from app.models.epi import ClimateObservation, DengueCase
from app.services.real_data_service import ingest_real_data

Base.metadata.create_all(bind=engine)
with SessionLocal() as db:
    ensure_geo_units(db)
    ensure_data_sources(db)
    db.commit()

    real_cases = db.scalar(select(func.count()).select_from(DengueCase).where(DengueCase.is_synthetic.is_(False))) or 0
    real_climate = db.scalar(select(func.count()).select_from(ClimateObservation).where(ClimateObservation.is_synthetic.is_(False))) or 0

    if real_cases == 0 or real_climate == 0:
        print("Descargando datos reales MINSA + NASA POWER (2015-2024)...")
        res = ingest_real_data(db, start_year=2015, end_year=2024, actor="colab_nested_cv")
        print("Ingesta completada:", res.get("effective_data_mode"))
    else:
        print(f"Datos reales verificados en SQLite: {real_cases} casos y {real_climate} registros climáticos.")

    panel = load_panel(db, mode="real")
    assert not panel.is_synthetic and panel.data_mode == "real", "ERROR: El panel no es 100 % real."
    print(f"Panel activo: {len(panel.panel)} filas | {len(panel.districts)} distritos | {len(panel.weeks)} semanas ({panel.weeks[0]} a {panel.weeks[-1]})")

# Respaldar dengai.db en Drive para futuras reconexiones instantáneas
if local_db.exists() and not drive_db.exists():
    shutil.copy2(local_db, drive_db)
    print(f"Respaldo guardado en Drive: {drive_db}")

In [ ]:
# 5. (Opcional recomendado) Prueba rápida de humo (~3 seg) antes de la corrida completa
import tempfile

with tempfile.TemporaryDirectory() as tmpdir:
    smoke_out = Path(tmpdir) / "smoke_nested_cv.json"
    cmd = [
        sys.executable, "training/run_nested_cv.py",
        "--models", "xgboost",
        "--horizon", "1",
        "--trials", "1",
        "--epochs", "1",
        "--data-mode", "real",
        "--gpu",
        "--output", str(smoke_out),
        "--telemetry-dir", tmpdir,
    ]
    subprocess.run(cmd, cwd=str(REPO_DIR), check=True)
print("Smoke test OK: GPU, datos reales y aserciones antifuga verificados.")

In [ ]:
# 6. Ejecutar Nested Walk-Forward CV completo (guardando cada fold directo en Google Drive)
# Puedes reducir MODELS a un subconjunto o ejecutar en varias sesiones gracias a --resume.
MODELS = ["xgboost", "lstm", "convlstm", "cnn_lstm", "convlstm_attention"]
HORIZON = 1
TRIALS = 50          # Anexo A recomienda 50 a 150 trials en fase final (usa 20 para corrida más rápida)
EPOCHS = 40          # Máximo de épocas (con early stopping patience=8)
OUTER_FOLDS = 5
INNER_FOLDS = 3
INNER_VAL_WEEKS = 13
SEED = 42

cmd = [
    sys.executable, "-u", "training/run_nested_cv.py",
    "--models", *MODELS,
    "--horizon", str(HORIZON),
    "--trials", str(TRIALS),
    "--epochs", str(EPOCHS),
    "--outer-folds", str(OUTER_FOLDS),
    "--inner-folds", str(INNER_FOLDS),
    "--inner-val-weeks", str(INNER_VAL_WEEKS),
    "--seed", str(SEED),
    "--data-mode", "real",
    "--gpu",
    "--output", str(OUTPUT_JSON),
    "--telemetry-dir", str(TELEMETRY_DIR),
    "--resume",
]

print("Ejecutando:", " ".join(cmd))
proc = subprocess.Popen(cmd, cwd=str(REPO_DIR), stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
assert proc.stdout is not None
for line in proc.stdout:
    print(line, end="")
ret = proc.wait()
if ret != 0:
    raise RuntimeError(f"run_nested_cv.py terminó con código {ret}. Revisa logs arriba; los folds completados están a salvo en {OUTPUT_JSON}.")
print(f"\n¡Corrida completada! Resultados guardados en: {OUTPUT_JSON}")

In [ ]:
# 7. Inspeccionar auditoría antifuga y tabla resumen de los 5 outer folds (Anexo A)
import json
import pandas as pd

report = json.loads(OUTPUT_JSON.read_text(encoding="utf-8"))
print("Auditoría antifuga:", json.dumps(report["leakage_audit"], indent=2, ensure_ascii=False))

summary_rows = []
fold_rows = []
for algo, entry in report["models"].items():
    row = {"modelo": algo, "estado": entry["status"], "folds": entry["n_outer_folds_completed"]}
    for metric, stats_dict in entry.get("summary", {}).items():
        row[f"{metric}_mean"] = stats_dict.get("mean")
        row[f"{metric}_std"] = stats_dict.get("std")
        row[f"{metric}_ci95"] = f"[{stats_dict.get('ci_95_lower')}, {stats_dict.get('ci_95_upper')}]"
    summary_rows.append(row)

    for f in entry.get("outer_folds", []):
        fold_rows.append({
            "modelo": algo,
            "outer_fold": f["outer_fold"],
            "outer_val_start": f["outer_val_origin_range"][0],
            "outer_val_end": f["outer_val_origin_range"][1],
            **f["metrics"],
        })

df_summary = pd.DataFrame(summary_rows)
df_folds = pd.DataFrame(fold_rows)
display(df_summary[["modelo", "estado", "folds", "MAE_mean", "MAE_std", "RMSE_mean", "MASE_mean", "WIS_mean", "coverage_mean", "AUC_PR_mean", "AUC_ROC_mean", "F1_mean", "MCC_mean", "Brier_mean", "tiempo_s_mean"]])

In [ ]:
# 8. Empaquetar resultados y disparar descarga automática (.zip) + copia en Drive
export_dir = Path("/content/paquete_nested_cv_fase2") if IN_COLAB else (REPO_DIR / "data" / "reports" / "phase2" / "export")
if export_dir.exists():
    shutil.rmtree(export_dir)
export_dir.mkdir(parents=True, exist_ok=True)

shutil.copy2(OUTPUT_JSON, export_dir / "nested_cv_results.json")
df_summary.to_csv(export_dir / "nested_cv_summary.csv", index=False)
df_folds.to_csv(export_dir / "nested_cv_per_fold.csv", index=False)
if TELEMETRY_DIR.exists():
    shutil.copytree(TELEMETRY_DIR, export_dir / "telemetry", dirs_exist_ok=True)

zip_base = Path("/content/resultados_nested_cv_fase2") if IN_COLAB else (REPO_DIR / "resultados_nested_cv_fase2")
zip_path = Path(shutil.make_archive(str(zip_base), "zip", export_dir))
drive_zip = DRIVE_DIR / zip_path.name
shutil.copy2(zip_path, drive_zip)
print(f"Paquete ZIP generado: {zip_path}")
print(f"Copia persistida en Drive: {drive_zip}")

if IN_COLAB:
    from google.colab import files
    files.download(str(zip_path))